In [6]:
%%time
import pandas as pd
import numpy as np
from pathlib import Path


CPU times: total: 0 ns
Wall time: 0 ns


In [7]:
%%time
data_path = Path('data')
df_name = 'training.csv'
df_path = data_path / df_name
df_path.absolute()
df = pd.read_csv(df_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72983 entries, 0 to 72982
Data columns (total 34 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   RefId                              72983 non-null  int64  
 1   IsBadBuy                           72983 non-null  int64  
 2   PurchDate                          72983 non-null  object 
 3   Auction                            72983 non-null  object 
 4   VehYear                            72983 non-null  int64  
 5   VehicleAge                         72983 non-null  int64  
 6   Make                               72983 non-null  object 
 7   Model                              72983 non-null  object 
 8   Trim                               70623 non-null  object 
 9   SubModel                           72975 non-null  object 
 10  Color                              72975 non-null  object 
 11  Transmission                       72974 non-null  obj

In [11]:
%%time


# Настройки отображения
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

def sec(title):
    print('\n' + '=' * 38)
    print(title)
    print('=' * 38)

def subsec(title):
    print('\n' + '-' * 38)
    print(title)
    print('-' * 38)

# --- Загрузка данных (если еще не загружено) ---
# data_path = Path('data')
# df_name = 'training.csv'
# df_path = data_path / df_name
# df = pd.read_csv(df_path)

sec('0. BASIC INFO & STRUCTURE')

print(f'Shape: {df.shape}')
print(f'Total rows: {len(df)}')
print(f'Total columns: {df.shape[1]}')
print(f'Memory usage (MB): {round(df.memory_usage(deep=True).sum() / 1024**2, 2)}')

# Список колонок с типами
print("\nColumns and dtypes:")
for i, col in enumerate(df.columns):
    null_count = df[col].isna().sum()
    dtype = str(df[col].dtype)
    print(f'{i:>3} | {col:<40} | {dtype:<10} | Nulls: {null_count}')


sec('1. TARGET VARIABLE ANALYSIS: IsBadBuy')

target = 'IsBadBuy'
if target in df.columns:
    print(f'Dtype: {df[target].dtype}')
    print(f'Unique values: {sorted(df[target].unique())}')

    vc = df[target].value_counts(dropna=False)
    print('\nValue counts:')
    print(vc.to_string())

    print('\nClass balance (%):')
    print((vc / len(df) * 100).round(2).to_string())

    # Проверка на дисбаланс
    bad_rate = df[target].mean()
    print(f'\nBase rate of Bad Buys: {bad_rate:.4f}')
else:
    print("Target column 'IsBadBuy' not found!")


sec('2. DATE COLUMN ANALYSIS: PurchDate')

date_col = 'PurchDate'
if date_col in df.columns:
    print(f'Raw dtype: {df[date_col].dtype}')
    print(f'Sample raw values:\n{df[date_col].head(5).tolist()}')

    # Парсинг даты
    parsed_dates = pd.to_datetime(df[date_col], errors='coerce')
    invalid_dates = df[date_col].notna() & parsed_dates.isna()

    print(f'\nParsed successfully: {parsed_dates.notna().sum()}')
    print(f'Invalid/Unparseable non-null: {invalid_dates.sum()}')

    if parsed_dates.notna().any():
        min_date = parsed_dates.min()
        max_date = parsed_dates.max()
        n_unique_days = parsed_dates.dt.normalize().nunique()

        print(f'Date range: {min_date.date()} to {max_date.date()}')
        print(f'Number of unique days: {n_unique_days}')

        # Распределение по месяцам для понимания временного ряда
        tmp_df = df.copy()
        tmp_df['PurchaseMonth'] = parsed_dates.dt.to_period('M')

        monthly_stats = tmp_df.groupby('PurchaseMonth').agg(
            count=('IsBadBuy', 'size'),
            bad_buy_rate=('IsBadBuy', 'mean')
        ).reset_index()

        print('\nMonthly stats (Count and Bad Buy Rate):')
        print(monthly_stats.head(10).to_string(index=False)) # Первые 10 месяцев
        print("...")
        print(monthly_stats.tail(10).to_string(index=False)) # Последние 10 месяцев

        # Сохраняем распарсенную дату обратно в df для следующих шагов
        # Важно: в реальном пайплайне мы делаем это один раз при загрузке
        df['PurchDate_dt'] = parsed_dates
else:
    print("Column 'PurchDate' not found.")


sec('3. MISSING VALUES OVERVIEW')

missing_summary = df.isna().sum()
missing_pct = (missing_summary / len(df) * 100).round(2)

miss_df = pd.DataFrame({
    'Missing_Count': missing_summary,
    'Missing_Percent': missing_pct
}).sort_values(by='Missing_Count', ascending=False)

# Показываем только те колонки, где есть пропуски
miss_with_nan = miss_df[miss_df['Missing_Count'] > 0]

print(f'Rows with at least one NaN: {df.isna().any(axis=1).sum()}')
print(f'Columns with NaNs: {len(miss_with_nan)}')

if not miss_with_nan.empty:
    print(miss_with_nan.to_string())
else:
    print('No missing values found.')


sec('4. NUMERIC FEATURES EXPLORATION')

# Исключаем таргет, ID и дату из числовых признаков для описания
exclude_cols = [target, 'RefId', 'PurchDate', 'PurchDate_dt', 'BYRNO']
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
feature_num_cols = [c for c in num_cols if c not in exclude_cols]

print(f'Numeric feature columns ({len(feature_num_cols)}):')
print(feature_num_cols)

if feature_num_cols:
    # ИСПРАВЛЕНО: убрана лишняя точка в конце строки
    desc = df[feature_num_cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

    # Добавляем информацию о уникальных значениях и нулях
    desc['nunique'] = df[feature_num_cols].nunique()
    desc['zeros'] = (df[feature_num_cols] == 0).sum()
    desc['negatives'] = (df[feature_num_cols] < 0).sum()
    desc['skewness'] = df[feature_num_cols].skew()

    print('\nDescriptive Statistics:')
    print(desc.to_string())

    # Корреляция с таргетом
    print('\nCorrelation with Target (IsBadBuy):')
    corr_target = df[feature_num_cols].corrwith(df[target]).sort_values(ascending=False)
    print(corr_target.to_string())

    # Топ корреляций между фичами (для поиска мультиколлинеарности)
    print('\nTop Absolute Correlations between Features (> 0.7):')
    corr_matrix = df[feature_num_cols].corr().abs()
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    pairs = corr_matrix.where(mask).stack().sort_values(ascending=False)
    high_corr_pairs = pairs[pairs > 0.7]
    print(high_corr_pairs.head(20).to_string())
else:
    print('No numeric features found.')


0. BASIC INFO & STRUCTURE
Shape: (72983, 34)
Total rows: 72983
Total columns: 34
Memory usage (MB): 72.65

Columns and dtypes:
  0 | RefId                                    | int64      | Nulls: 0
  1 | IsBadBuy                                 | int64      | Nulls: 0
  2 | PurchDate                                | object     | Nulls: 0
  3 | Auction                                  | object     | Nulls: 0
  4 | VehYear                                  | int64      | Nulls: 0
  5 | VehicleAge                               | int64      | Nulls: 0
  6 | Make                                     | object     | Nulls: 0
  7 | Model                                    | object     | Nulls: 0
  8 | Trim                                     | object     | Nulls: 2360
  9 | SubModel                                 | object     | Nulls: 8
 10 | Color                                    | object     | Nulls: 8
 11 | Transmission                             | object     | Nulls: 9
 12 | WheelTypeID